In [1]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp
import time

# coins = ["bitcoin", "ethereum", "solana"]
coins = ["bitcoin"]

all_data = []

for coin in coins:
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart?vs_currency=usd&days=1"
    
    for attempt in range(3):
        response = requests.get(url)
        if response.status_code == 200:
            break
        time.sleep(2)
    
    if response.status_code == 200:
        data = response.json()
        
        prices = data["prices"]  # [timestamp, price]
        
        for record in prices:
            all_data.append({
                "coin": coin,
                "timestamp": record[0],
                "price": record[1]
            })
    else:
        print(f"Failed for {coin}")

In [2]:
df = pd.DataFrame(all_data)

In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

In [4]:
df.head()

,coin,timestamp,price
0,bitcoin,2026-03-24 23:04:51.199,70270.614982
1,bitcoin,2026-03-24 23:09:55.451,70390.331557
2,bitcoin,2026-03-24 23:14:51.764,70518.611249
3,bitcoin,2026-03-24 23:19:51.644,70496.192510
4,bitcoin,2026-03-24 23:24:56.071,70507.414839


In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

spark_df = spark.createDataFrame(df)
spark_df = spark_df.withColumn("ingestion_time", current_timestamp())
spark_df.show(5)

/usr/local/spark/python/pyspark/sql/pandas/conversion.py:474: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
/usr/local/spark/python/pyspark/sql/pandas/conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


+-------+--------------------+-----------------+--------------------+
|   coin|           timestamp|            price|      ingestion_time|
+-------+--------------------+-----------------+--------------------+
|bitcoin|2026-03-24 23:04:...|70270.61498164976|2026-03-25 23:06:...|
|bitcoin|2026-03-24 23:09:...|70390.33155685537|2026-03-25 23:06:...|
|bitcoin|2026-03-24 23:14:...|70518.61124884721|2026-03-25 23:06:...|
|bitcoin|2026-03-24 23:19:...|70496.19251036912|2026-03-25 23:06:...|
|bitcoin|2026-03-24 23:24:...|70507.41483883654|2026-03-25 23:06:...|
+-------+--------------------+-----------------+--------------------+
only showing top 5 rows



In [8]:
# spark_df.write.format("delta") \
#     .mode("append") \
#     .save("/home/jovyan/CryptoInsight")

In [10]:
# Improve Your Batch Design 
# Add: Partition by date
# 🔥 Why this matters:
# - Faster queries
# - Efficient incremental loads
# - Scalable storage

from pyspark.sql.functions import to_date

spark_df = spark_df.withColumn("date", to_date("timestamp"))

spark_df.show(5)

+-------+--------------------+-----------------+--------------------+----------+
|   coin|           timestamp|            price|      ingestion_time|      date|
+-------+--------------------+-----------------+--------------------+----------+
|bitcoin|2026-03-24 23:04:...|70270.61498164976|2026-03-25 23:11:...|2026-03-24|
|bitcoin|2026-03-24 23:09:...|70390.33155685537|2026-03-25 23:11:...|2026-03-24|
|bitcoin|2026-03-24 23:14:...|70518.61124884721|2026-03-25 23:11:...|2026-03-24|
|bitcoin|2026-03-24 23:19:...|70496.19251036912|2026-03-25 23:11:...|2026-03-24|
|bitcoin|2026-03-24 23:24:...|70507.41483883654|2026-03-25 23:11:...|2026-03-24|
+-------+--------------------+-----------------+--------------------+----------+
only showing top 5 rows



In [15]:
spark_df.write.partitionBy("coin") \
    .format("delta") \
    .mode("append") \
    .save("/mnt/bronze/crypto_prices")

# spark_df.write.option("header", "true").format("csv").save("bronze")

In [ ]:
OPTIMIZE delta.`/mnt/silver/crypto`
ZORDER BY (symbol)